# Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 3. 

In [2]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 3 - arXiv abstracts

This dataset contains arXiv abstracts over a many year. Let's first load it and examine the fields:

In [3]:
from datasets import load_dataset

# Load the arXiv abstracts dataset
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")

print(dataset)

Dataset({
    features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'abstract', 'report-no', 'categories', 'versions'],
    num_rows: 1999486
})


We can now filter it to get abstracts about a particular topic or field:

In [4]:
# -----------------------
# Optional: filter by category
# -----------------------
field = "cs.LG"  # Example: Machine Learning
def filter_category(example):
    return field in example['categories']

filtered = dataset.filter(filter_category)

# -----------------------
# Remove missing/short abstracts
# -----------------------
def valid_abstract(example):
    return example['abstract'] is not None and len(example['abstract']) > 50

filtered = filtered.filter(valid_abstract)

Another useful step is to get the date of each abstract, e.g., for plotting trends. This is not stored in the dataset object, but we can infer it from the ID. 

In [5]:
# -----------------------
# Infer year from arXiv ID
# -----------------------
def extract_year(arxiv_id):
    if arxiv_id.startswith("cs/"):
        arxiv_id = arxiv_id.split("/")[1]
    yy = int(arxiv_id[:2])
    return 1900 + yy if yy >= 91 else 2000 + yy

years = [extract_year(x) for x in filtered['id']]

# -----------------------
# Prepare docs
# -----------------------
docs = filtered['abstract']